# 2교시 | Transformer 핵심 구조
**환경**: Google Colab | **방법**: 이론 + 실습

---

## 학습 목표

- Transformer의 전체 구성 요소를 구조도와 함께 설명할 수 있다.
- GPT-2 모델을 직접 로드하여 레이어 수, 헤드 수, 임베딩 차원을 확인한다.
- BPE(Byte Pair Encoding) 토크나이저의 서브워드 분할 방식을 관찰한다.

### 처음 읽는 분을 위한 안내

- 이번 교시를 한 줄로 보면: Transformer가 문장 전체를 보며 토큰 관계를 계산하는 구조를 처음으로 잡는 시간이다.
- 처음에는 여기까지 이해하면 충분하다: 토큰이 벡터가 되고, Attention으로 서로를 참고하며, 그 결과로 다음 토큰 확률이 계산된다는 흐름만 잡아도 된다.
- 헷갈려도 괜찮은 부분: Q, K, V 수식이나 `d_model` 같은 숫자는 처음부터 완벽히 외울 필요 없다. 먼저 "누구를 얼마나 참고하는가"라는 뜻이 더 중요하다.

### 용어 미니사전

| 용어 | 아주 쉽게 말하면 |
|---|---|
| 토큰(Token) | 문장을 모델이 다루기 좋게 나눈 작은 조각 |
| 임베딩(Embedding) | 토큰을 숫자 벡터로 바꾼 표현 |
| Attention | 어떤 토큰을 더 많이 참고할지 정하는 계 |
| Head | Attention을 서로 다른 관점으로 한 번씩 보는 작은 창 |
| `d_model` | 토큰 벡터의 길이, 즉 표현 차원 |
| LM Head(Language Modeling Head) | 마지막 벡터를 다음 토큰 점수(logit)로 바꾸는 층 |

---

## 이론

### Transformer 등장 배경

2017년 Google Brain의 논문 *"Attention Is All You Need"* (Vaswani et al.)에서 제안된 아키텍처로, 기존 RNN/LSTM의 두 가지 한계를 극복했다.

| 기존 RNN/LSTM의 한계 | Transformer의 해결 |
|---|---|
| 순차 처리 → 병렬화 불가 | 전체 시퀀스를 동시에 처리 |
| 장거리 의존성 소멸 | Attention으로 임의 거리 직접 연결 |

이후 자연어 처리를 넘어 이미지(ViT), 음성, 단백질 구조 예측까지 확장된 범용 아키텍처가 되었다.

**먼저 큰 그림부터 이해하기**

Transformer는 한 문장을 읽을 때, 각 단어를 따로따로 보는 것이 아니라 **문장 전체 안에서 서로 어떤 관련이 있는지 동시에 계산**한다.  
예를 들어, 문장 `The animal didn't cross the street because it was tired.` 에서 `it`이 무엇을 가리키는지 이해하려면 앞쪽 단어들과의 관계를 봐야 한다. Transformer는 바로 이 "관계 계산"을 Attention으로 수행한다.

즉, Transformer의 핵심은 아래 한 줄로 요약할 수 있다.

> "각 토큰을 벡터로 바꾸고, 다른 토큰들과의 관련도를 계산한 뒤, 그 정보를 여러 층에서 반복적으로 정제한다."

### 입력에서 출력까지 한 번에 보기

학습자가 가장 많이 헷갈리는 지점은 "지금 모델 안에서 무엇이 순서대로 일어나는가"이다. 아래 순서를 먼저 머릿속에 넣고 세부 요소를 보면 이해가 훨씬 쉽다.

1. 문장을 토크나이저(BPE알고리즘)가 잘게 나눠 토큰 ID로 바꾼다.-토큰화
2. 각 토큰 ID를 임베딩 벡터로 바꾼다. - 임베딩
3. 토큰의 순서를 알 수 있도록 위치 정보를 더한다.- PE(Positional Encoding)
4. 각 토큰이 다른 토큰을 얼마나 참고해야 하는지 Self-Attention으로 계산한다.-MHA
5. 여러 Attention Head가 서로 다른 관점의 관계를 본다. -MHA
6. 결과를 합친 뒤 MLP(Feed-Forward Network)로 한 번 더 가공한다. -비선혀함수(GELU)
7. 이 블록을 여러 번 반복하면서 표현이 점점 정교해진다. - multi layer
8. 마지막 벡터로 다음 토큰의 확률을 계산한다. - LM Head(logit vector softemax이요 확률)( logits -> token -> propability->softemax)

### 핵심 구성 요소

아래 그림은 "텍스트가 모델 안에서 어떤 순서로 변환되는지"를 보여준다.

```
입력 문장
  ↓
[Tokenizer] 텍스트 → 서브워드 → 정수 ID
  ↓
[Token Embedding] 정수 ID → d_model 차원 벡터
  ↓
[Positional Encoding] 위치 정보 주입(더하기)
  ↓
[Transformer Block] × N회 반복
  ├─ Multi-Head Self-Attention
  ├─ Add & Layer Norm (잔차추가 - 정규화 Layer Norm대신 요즘은 RMS 사용)
  ├─ Feed-Forward Network (2-layer MLP)
  └─ Add & Layer Norm
  ↓
[LM Head(Language Modeling Head)] d_model → vocab_size 선형 변환(벡터를 토큰으로 언임베딩해서 변환)
  ↓
[Softmax] 다음 토큰 확률 분포
```

[Transformer](https://wikidocs.net/156986)
  

**각 구성 요소의 역할**

| 구성 요소 | 역할 |
|---|---|
| Token Embedding | 단어/서브워드 → 고차원 연속 벡터 변환 |
| Positional Encoding | 순서 정보 주입 (sin/cos 또는 학습 가능) |
| Multi-Head Attention | 토큰 간 관계를 여러 관점에서 병렬 계산 |
| Feed-Forward Network | 각 위치에서 독립적으로 비선형 변환 |
| Layer Normalization | 학습 안정화 (Pre-LN: 최신 모델의 표준) |
| Residual Connection | 그래디언트 소실 방지, 하위 레이어 정보 보존 |

### Self-Attention을 직관적으로 이해하기

Self-Attention은 "현재 토큰이 문장 안의 어떤 다른 토큰을 얼마나 참고할지 결정하는 과정"이다.

예를 들어 문장 `The cat sat on the mat`에서 `sat`을 이해할 때 모델은 `cat`, `on`, `mat` 같은 다른 토큰들을 함께 본다. 이때 모든 토큰을 똑같이 보는 것이 아니라, **현재 토큰과 더 관련 있는 토큰에 더 큰 가중치**를 둔다.

이 과정에서 자주 나오는 세 가지 벡터가 있다.(문맥만들기 위해)

| 기호 | 직관적 의미 |
|---|---|
| Q (Query) | "나는 지금 어떤 정보를 찾고 있나?" |
| K (Key) | "나는 어떤 특징을 가진 토큰인가?" |
| V (Value) | "실제로 전달할 정보는 무엇인가?" |

동작 순서는 다음처럼 이해하면 된다.

1. 각 토큰에서 Q, K, V를 만든다.
2. 현재 토큰의 Q와 다른 토큰들의 K를 비교해 관련도를 계산한다.
3. 관련도가 큰 토큰의 V를 더 많이 반영한다.
4. 이렇게 얻은 결과가 "문맥이 반영된 새로운 토큰 표현"이 된다.

수식은 아래처럼 표현되지만, 처음에는 외우기보다 의미를 이해하는 것이 중요하다.

$$
Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

- $QK^T$: 토큰 간 유사도 점수
- $\sqrt{d_k}$로 나누기: 값이 너무 커지는 것을 막아 학습을 안정화  
   d_k : d_model을 head로 나눈값
- `softmax`: 점수를 확률처럼 바꿔 "무엇을 얼마나 볼지" 결정

### 왜 Multi-Head가 필요한가

Head가 하나뿐이면 문장 관계를 한 가지 방식으로만 보게 된다. 하지만 실제 문장에는 여러 종류의 관계가 섞여 있다.

- 문법 관계: 주어-동사 연결
- 의미 관계: 어떤 단어가 어떤 대상을 설명하는지
- 거리 관계: 가까운 단어를 보는지, 먼 단어를 보는지

Multi-Head Attention은 이 관계들을 **여러 시선으로 병렬 관찰**하게 해 준다.  
즉, 한 Head는 가까운 문법 정보를 보고, 다른 Head는 먼 거리의 의미 연결을 볼 수 있다.

### Feed-Forward, Residual, LayerNorm은 왜 필요한가

Attention만으로는 충분하지 않다. Attention이 "어디를 볼지"를 결정한다면, 그 뒤의 구성 요소들은 "얻은 정보를 어떻게 안정적으로 가공할지"를 담당한다.

| 구성 요소 | 쉽게 이해하면 |
|---|---|
| Feed-Forward Network | 각 토큰 벡터를 더 풍부하게 변환하는 작은 MLP |
| Residual Connection | 기존 정보를 우회로로 같이 전달해 정보 손실을 줄이는 장치 |
| LayerNorm | 값의 분포를 안정화해 학습이 흔들리지 않게 하는 장치 |

특히 Residual Connection은 깊은 네트워크에서 매우 중요하다. 층이 많아져도 이전 정보가 완전히 사라지지 않도록 도와주기 때문이다.

### Encoder vs Decoder

```
Encoder (BERT 계열)          Decoder-only (GPT 계열)
────────────────────────     ─────────────────────────────
입력 전체를 동시에 참조        이전 토큰만 참조 (Causal Mask)
문장 이해·분류에 적합          텍스트 생성에 적합
양방향 Self-Attention         단방향 Self-Attention
```

정리하면 다음과 같다.

- Encoder는 "문장을 잘 이해하는 일"에 강하다.
- Decoder-only는 "문장을 한 토큰씩 생성하는 일"에 강하다.
- 최근 LLM은 대부분 Decoder-only를 기반으로 발전했다.

### Decoder-only 모델은 어떻게 다음 토큰을 생성할까

GPT 계열은 Decoder-only 구조를 사용한다. 여기서 핵심은 **미래 토큰을 미리 보면 안 된다**는 점이다.

예를 들어 `나는 오늘 학교에` 다음 단어를 예측할 때, 정답인 `간다`를 미리 보면 학습이 성립하지 않는다. 그래서 GPT는 **Causal Mask**를 사용해 현재 위치보다 뒤쪽 토큰을 보지 못하게 막는다.

즉, 각 위치는 다음처럼 동작한다.

- 1번째 토큰: 자기 자신만 참고
- 2번째 토큰: 1~2번째 참고
- 3번째 토큰: 1~3번째 참고
- 마지막 토큰: 앞의 모든 토큰 참고

이 구조 덕분에 GPT는 "이전 문맥만 보고 다음 토큰을 예측하는 모델"이 된다.

### GPT-2 모델 스펙

| 모델 | 레이어 수 | 헤드 수 | d_model | 파라미터 |
|---|---|---|---|---|
| GPT-2 Small | 12 | 12 | 768 | 128M |
| GPT-2 Medium | 24 | 16 | 1024 | 345M |
| GPT-2 Large | 36 | 20 | 1280 | 774M |
| GPT-2 XL | 48 | 25 | 1600 | 1.5B |

> Colab T4에서는 GPT-2 Small을 사용한다.

여기서 교육생이 꼭 연결해서 이해해야 할 포인트는 아래 두 가지다.

- 레이어 수가 많을수록 더 깊게 문맥을 정제할 수 있지만 계산량이 늘어난다.
- `d_model`이 클수록 더 풍부한 표현이 가능하지만 메모리 사용량도 커진다.

### BPE (Byte Pair Encoding) 토크나이저

GPT 계열은 BPE 방식으로 텍스트를 서브워드 단위로 분할한다.

- 자주 등장하는 문자 쌍을 반복적으로 합쳐 어휘를 구성
- 미등록 단어(OOV)가 없음 — 어떤 단어도 서브워드로 분해 가능
- 예: `"Transformer"` → `['Trans', 'former']`
- 예: `"ChatGPT"` → `['Chat', 'G', 'PT']`

왜 굳이 단어 전체가 아니라 서브워드 단위로 자를까?

- 단어 전체만 쓰면 어휘 사전이 너무 커진다.
- 글자 단위만 쓰면 시퀀스가 너무 길어진다.
- 서브워드는 그 중간 지점이라서, **어휘 크기와 표현력을 적절히 균형** 잡을 수 있다.

특히 영어와 한국어가 섞인 문장, 신조어, 고유명사처럼 처음 보는 표현도 어느 정도 처리할 수 있다는 점이 실전에서 중요하다.

### 이론을 읽을 때 꼭 잡아야 할 연결 고리

아래 네 문장을 스스로 설명할 수 있으면 이번 차시 핵심을 이해한 것이다.

1. 토큰은 먼저 ID가 되고, 그다음 벡터가 된다.
2. 벡터만으로는 순서를 모르기 때문에 위치 정보가 필요하다.
3. Self-Attention은 각 토큰이 다른 토큰을 얼마나 참고할지 계산한다.
4. Transformer Block을 여러 번 반복하면 문맥을 반영한 표현이 점점 정교해진다.

---

## 실습

실습은 아래 순서대로 **한 단계씩 실행**하는 것을 권장한다.  
각 코드 블록은 바로 다음 설명과 연결되므로, 한 셀을 실행한 뒤 출력 결과를 확인하고 다음 셀로 넘어가면 된다.

### 자주 막히는 오류

- `transformers` import 오류: 1교시 설치 셀이 현재 Colab 세션에서 다시 실행되었는지 먼저 확인한다.
- 모델 로딩이 오래 걸림: 첫 실행에서는 다운로드와 캐시 생성 때문에 시간이 더 걸릴 수 있다.
- 영어 예문 토큰화가 예상과 다름: 토크나이저는 사람이 생각하는 단어 단위와 다르게 잘릴 수 있으며, 이것이 정상 동작이다.
- shape 출력이 낯설다: `(batch, seq_len, hidden_size)` 순서라고 먼저 읽으면 해석이 쉬워진다.

### 실패했을 때 체크 순서

1. 패키지와 모델 준비를 확인한다: `transformers`, `torch`, 모델 로딩 셀이 먼저 실행됐는지 본다.
2. 출력 shape와 토큰 리스트를 먼저 읽는다: 숫자가 이상해 보여도 `(batch, seq_len, hidden_size)`와 토큰 분할 결과를 차분히 확인한다.
3. 막힌 단계만 다시 실행한다: 계속 꼬이면 Step 0부터 다시 실행해 의존 변수를 복구한다.

### Step 0. 실습 준비

먼저 모델과 토크나이저를 불러오기 위한 기본 패키지를 import한다.

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch



이 단계에서 확인할 점:

- 오류 없이 import가 되는가?
- Colab이라면 런타임이 정상적으로 연결되어 있는가?

### Step 1. GPT-2 모델과 토크나이저 로드

이번 실습에서는 `gpt2`를 사용한다.  
`tokenizer`는 텍스트를 토큰 ID로 바꾸고, `model`은 그 토큰을 벡터로 처리하는 역할을 한다.

In [2]:
# 실습용 모델 이름
model_name = "gpt2"

# 토크나이저 로드: 텍스트를 토큰 ID로 바꾸는 도구
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 모델 로드: GPT-2의 본체
# output_attentions=True를 주면 나중에 attention 정보도 함께 볼 수 있다.
model = AutoModel.from_pretrained(model_name, output_attentions=True)

# 추론 모드로 전환: dropout 등 학습 전용 동작을 끈다.
model.eval()

print("모델과 토크나이저 로드 완료")
print(f"모델명: {model_name}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

모델과 토크나이저 로드 완료
모델명: gpt2




이 단계에서 확인할 점:

- 다운로드가 정상적으로 끝나는가?
- `모델과 토크나이저 로드 완료`가 출력되는가?

### Step 2. 모델 설정(Config) 읽기

이번 단계에서는 GPT-2가 몇 개의 레이어와 헤드를 가지고 있는지 확인한다.  
이 값들은 이론에서 본 `n_layer`, `n_head`, `d_model`, `vocab_size`와 직접 연결된다.

In [3]:
# 모델 내부 설정값(config) 읽기
cfg = model.config

print("=" * 50)
print(f"모델명            : {model_name}")
print(f"레이어 수          : {cfg.n_layer}")
print(f"어텐션 헤드 수     : {cfg.n_head}")
print(f"임베딩 차원 d_model: {cfg.n_embd}")
print(f"어휘 크기          : {cfg.vocab_size:,}")
print(f"최대 시퀀스 길이    : {cfg.n_positions}")
print("=" * 50)

모델명            : gpt2
레이어 수          : 12
어텐션 헤드 수     : 12
임베딩 차원 d_model: 768
어휘 크기          : 50,257
최대 시퀀스 길이    : 1024




이 단계에서 확인할 점:

- 레이어 수가 12, 헤드 수가 12로 나오는가?
- `d_model=768`, `vocab_size=50,257`이 출력되는가?

### Step 3. 전체 파라미터 수 계산

모델이 얼마나 큰지 감각을 잡기 위해 전체 파라미터 수를 계산한다.  
GPT-2 Small은 약 124M 파라미터를 가진다.

In [4]:
# 모델의 모든 파라미터 개수를 합산
total_params = sum(p.numel() for p in model.parameters())

print(f"전체 파라미터 수: {total_params:,}")
print(f"약 {total_params / 1e6:.1f}M parameters")

전체 파라미터 수: 124,439,808
약 124.4M parameters




이 단계에서 확인할 점:

- 약 `124M` 수준으로 출력되는가?
- 이 값이 위의 GPT-2 Small 스펙 표와 연결되는가?

### Step 4. 모델의 큰 구조 살펴보기

이제 모델이 내부적으로 어떤 큰 블록들로 이루어져 있는지 확인한다.  
처음에는 세부 구현보다, "임베딩 층이 있고, Transformer 블록이 있고, 마지막 정규화 층이 있구나" 정도를 읽어내면 충분하다.

In [5]:
print("\n[모델 구조 요약]")

# named_children()는 모델의 바로 아래 단계 모듈들을 보여준다.
for name, module in model.named_children():
    # 각 하위 모듈이 가진 파라미터 수를 함께 계산해서 출력
    module_params = sum(p.numel() for p in module.parameters())
    print(f"  {name:<10}: {type(module).__name__:<25} ({module_params / 1e6:.1f}M params)")


[모델 구조 요약]
  wte       : Embedding                 (38.6M params)
  wpe       : Embedding                 (0.8M params)
  drop      : Dropout                   (0.0M params)
  h         : ModuleList                (85.1M params)
  ln_f      : LayerNorm                 (0.0M params)




이 단계에서 확인할 점:

- `wte`, `wpe`, `h`, `ln_f` 같은 이름이 보이는가?
- `h`가 Transformer 블록 묶음이라는 점을 연결할 수 있는가?

### Step 5. 토크나이저가 문장을 어떻게 자르는지 관찰하기

이번 단계에서는 BPE 토크나이저가 실제 문장을 어떤 토큰들로 나누는지 확인한다.  
영어 문장, 고유명사, 한국어가 섞인 문장을 비교해 보면 서브워드 분할의 감각을 잡기 좋다.

처음 결과를 보면 토큰 리스트가 낯설 수 있다. 특히 아래 세 가지 규칙을 먼저 알고 보면 훨씬 읽기 쉽다.

1. `Ġ`가 붙은 토큰은 "앞에 공백이 있었다"는 뜻이다. 예를 들어 `Ġarchitecture`는 실제로는 ` architecture`처럼 앞에 띄어쓰기가 포함된 토큰이다.
2. 영어는 자주 쓰이는 단위가 많이 학습되어 있어, `Trans + former`, `revolution + ized`처럼 사람이 읽을 수 있는 서브워드 단위로 비교적 깔끔하게 나뉘는 경우가 많다.
3. 한국어는 GPT-2가 영어 중심 데이터로 학습된 바이트 단위 BPE(Byte Pair Encoding)를 쓰기 때문에, 글자 단위가 아니라 바이트 조각처럼 더 잘게 쪼개져 `ìĿ`, `Ģ` 같은 낯선 형태로 보일 수 있다.

즉, 이번 출력에서는 "토큰이 예쁘게 보이는가"보다 "영어는 비교적 큰 서브워드 단위로, 한국어는 더 잘고 불규칙한 단위로 분해되는가"를 읽어내는 것이 핵심이다.

In [6]:
print("\n[토크나이저 실험]")

test_texts = [
    "The Transformer architecture changed everything in NLP.",
    "ChatGPT revolutionized human-computer interaction.",
    "Tokenization은 서브워드 분할 방식을 사용한다.",
]

for text in test_texts:
    # 텍스트를 모델 입력 형식으로 바꾸고, input_ids만 꺼낸다.
    input_ids = tokenizer(text, return_tensors="pt")["input_ids"][0]

    # 숫자 ID를 사람이 읽을 수 있는 토큰 문자열로 다시 변환한다.
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    print(f"\n입력 문장: {text}")
    print(f"토큰 ID : {input_ids.tolist()}")
    print(f"토큰    : {tokens}")
    print(f"토큰 개수: {len(tokens)}개")


[토크나이저 실험]

입력 문장: The Transformer architecture changed everything in NLP.
토큰 ID : [464, 3602, 16354, 10959, 3421, 2279, 287, 399, 19930, 13]
토큰    : ['The', 'ĠTrans', 'former', 'Ġarchitecture', 'Ġchanged', 'Ġeverything', 'Ġin', 'ĠN', 'LP', '.']
토큰 개수: 10개

입력 문장: ChatGPT revolutionized human-computer interaction.
토큰 ID : [30820, 38, 11571, 5854, 1143, 1692, 12, 33215, 10375, 13]
토큰    : ['Chat', 'G', 'PT', 'Ġrevolution', 'ized', 'Ġhuman', '-', 'computer', 'Ġinteraction', '.']
토큰 개수: 10개

입력 문장: Tokenization은 서브워드 분할 방식을 사용한다.
토큰 ID : [30642, 1634, 35975, 222, 23821, 226, 250, 167, 116, 234, 168, 249, 234, 167, 241, 250, 31619, 114, 226, 47991, 254, 31619, 108, 102, 168, 233, 251, 35975, 226, 23821, 8955, 168, 248, 102, 47991, 250, 46695, 97, 13]
토큰    : ['Token', 'ization', 'ìĿ', 'Ģ', 'Ġì', 'Ħ', 'ľ', 'ë', '¸', 'Į', 'ì', 'Ľ', 'Į', 'ë', 'ĵ', 'ľ', 'Ġë', '¶', 'Ħ', 'íķ', 'ł', 'Ġë', '°', '©', 'ì', 'ĭ', 'Ŀ', 'ìĿ', 'Ħ', 'Ġì', 'Ĥ¬', 'ì', 'ļ', '©', 'íķ', 'ľ', 'ëĭ', '¤', '.']
토큰 개수: 39개


이 단계에서 확인할 점:

- 영어 단어가 통째로 잘리는지, 또는 `Trans + former`, `revolution + ized`처럼 서브워드로 쪼개지는지 관찰
- `Ġ` 표기가 "공백 뒤에서 시작한 토큰"이라는 뜻으로 보이는지 확인
- 한국어가 글자 단위라기보다 바이트 조각에 가까운 작은 단위로 많이 분해되는지 확인
- 영어 문장보다 한국어가 섞인 문장에서 토큰 개수가 더 빠르게 늘어나는지 비교

### Step 6. 한 문장이 모델 안에서 어떤 shape로 바뀌는지 보기

이제 실제 문장을 모델에 넣어 `hidden_states`를 확인한다.  
이 단계의 핵심은 "토큰 수가 몇 개인지", "레이어를 거치며 몇 개의 hidden state가 생기는지", "각 hidden state의 shape가 무엇을 뜻하는지"를 읽는 것이다.

In [7]:
print("\n[Hidden State Shape 확인]")

sample_text = "Hello, Transformer!"

# 문장을 토큰 ID 텐서로 변환
inputs = tokenizer(sample_text, return_tensors="pt")

with torch.no_grad():
  # output_hidden_states=True를 주면 각 층의 출력을 모두 받을 수 있다.
  outputs = model(**inputs, output_hidden_states=True)

# 입력 토큰 수 확인
seq_len = inputs["input_ids"].shape[1]

# hidden_states는 (임베딩 출력 1개 + 각 레이어 출력들)로 구성된다.
num_hidden_states = len(outputs.hidden_states)

# 첫 번째 hidden state의 shape 확인
first_shape = outputs.hidden_states[0].shape

print(f"입력 문장        : {sample_text}")
print(f"입력 토큰 수      : {seq_len}")
print(f"hidden_states 개수: {num_hidden_states}")
print(f"첫 hidden state shape: {first_shape}")
print(
  f"→ (batch_size={first_shape[0]}, seq_len={first_shape[1]}, d_model={first_shape[2]})"
)


[Hidden State Shape 확인]
입력 문장        : Hello, Transformer!
입력 토큰 수      : 5
hidden_states 개수: 13
첫 hidden state shape: torch.Size([1, 5, 768])
→ (batch_size=1, seq_len=5, d_model=768)


이 단계에서 확인할 점:

- `입력 토큰 수: 5`가 왜 나왔는지 생각해 본다. 이것은 `Hello, Transformer!` 문장이 토크나이저를 거친 뒤 5개의 토큰으로 나뉘었다는 뜻이다.
- `hidden_states 개수: 13`이 왜 나왔는지 확인한다. GPT-2 Small은 12개 레이어를 가지므로, `임베딩 출력 1개 + 각 레이어 출력 12개 = 총 13개`로 이해하면 된다.
- `torch.Size([1, 5, 768])`를 세 숫자로 나눠 읽어 본다. `1`은 한 번에 넣은 문장 개수(batch size), `5`는 현재 문장의 토큰 개수(seq_len), `768`은 각 토큰이 가진 벡터 길이(`d_model`)이다.
- 특히 마지막 숫자 `768`은 "토큰 하나를 768차원 숫자 벡터로 표현한다"는 뜻이지, 토큰이 768개라는 뜻이 아니라는 점을 구분한다.
- 즉, 이번 출력은 "문장 1개가 토큰 5개로 나뉘었고, 각 토큰이 길이 768의 벡터로 표현되어 레이어마다 전달된다"고 문장으로 풀어 설명할 수 있으면 된다.

### Step 7. 첫 층과 마지막 층 표현 비교하기

마지막으로, 임베딩 직후 표현과 마지막 레이어 출력의 shape를 비교해 본다.  
shape는 같아도 내부 값의 의미는 달라진다는 점이 중요하다. 처음 표현은 "토큰의 기본 표현"에 가깝고, 마지막 표현은 "문맥이 반영된 표현"에 가깝다.

In [ ]:
embedding_output = outputs.hidden_states[0]
final_output = outputs.hidden_states[-1]

print("[첫 층 vs 마지막 층 비교]")
print(f"임베딩 층 shape   : {embedding_output.shape}")
print(f"마지막 층 shape   : {final_output.shape}")

# 두 텐서의 shape는 같지만, 값은 일반적으로 달라진다.
same_shape = embedding_output.shape == final_output.shape
print(f"shape가 같은가?   : {same_shape}")

[첫 층 vs 마지막 층 비교]
임베딩 층 shape   : torch.Size([1, 5, 768])
마지막 층 shape   : torch.Size([1, 5, 768])
shape가 같은가?   : True


이 단계에서 확인할 점:

- `shape가 같은가? : True`가 나왔다면, 임베딩 층과 마지막 층이 모두 같은 틀 `(batch_size, seq_len, d_model)`을 유지하고 있다는 뜻이다.
- 여기서 `같은 shape`는 "토큰 개수와 벡터 길이가 유지된다"는 뜻이지, 두 텐서 안의 값이나 의미가 같다는 뜻은 아니다.
- 첫 층의 `(1, 5, 768)`은 "문장 1개, 토큰 5개, 각 토큰의 초기 벡터 길이 768"로 읽고, 마지막 층의 `(1, 5, 768)`도 겉모양은 같지만 여러 레이어를 거치며 문맥이 반영된 결과라고 이해한다.
- 즉, 토큰 자리는 그대로 유지한 채 각 자리의 벡터 내용만 점점 더 정교하게 바뀐다고 생각하면 된다.
- 이번 출력은 "모델이 토큰 수를 늘리거나 줄이지 않고, 같은 5개 토큰에 대해 더 문맥적인 표현으로 계속 갱신한다"고 설명할 수 있으면 충분하다.

### 실습을 마친 뒤 정리

위 실습이 끝나면 아래 내용을 말로 설명할 수 있어야 한다.

1. GPT-2 Small은 12개 레이어, 12개 헤드, 768차원 임베딩을 사용한다.
2. 토크나이저는 문장을 서브워드 단위의 토큰 ID로 바꾼다.
3. 모델은 각 토큰을 `d_model` 차원의 벡터로 바꿔 처리한다.
4. 각 레이어를 거치며 문맥이 반영된 hidden state가 만들어진다.
5. 첫 층과 마지막 층의 shape는 같아도, 담고 있는 정보는 다르다.

**기대 출력**
```
=============================================
모델명         : gpt2
레이어 수       : 12
어텐션 헤드 수  : 12
임베딩 차원(d)  : 768
어휘 크기       : 50,257
최대 시퀀스 길이 : 1024
=============================================
전체 파라미터   : 124M
...
```

### 관찰 과제

1. `vocab_size`가 50,257인 이유는? (50,000 BPE 병합 + 256 바이트 + 1 EOS)
2. 한국어 텍스트를 입력했을 때 토큰이 어떻게 분리되는지 확인
3. `hidden_states[0]`(임베딩 층)과 `hidden_states[-1]`(마지막 층)의 shape 비교

---

## 핵심 개념 정리

| 용어 | 의미 |
|---|---|
| `d_model` | 토큰 임베딩의 차원 수 (GPT-2: 768) |
| `n_heads` | Multi-Head Attention의 헤드 수 |
| `n_layers` | Transformer 블록의 반복 횟수 |
| `vocab_size` | 토크나이저가 인식하는 전체 토큰 수 |
| `d_head(dk)` | 각 헤드의 차원 = d_model / n_heads |

### 혼자 점검하기

아래 질문에 막힘 없이 답하면 이론 이해가 잘 된 것이다.

1. Transformer가 RNN보다 병렬 처리에 유리한 이유는 무엇인가?
2. Positional Encoding이 없으면 모델에 어떤 문제가 생기는가?
3. Self-Attention에서 Q, K, V는 각각 어떤 역할을 하는가?
4. Multi-Head Attention은 Head가 1개인 경우보다 무엇이 좋은가?
5. GPT가 미래 토큰을 보지 못하도록 막는 이유는 무엇인가?

---

## 다음 교시 예고

**3교시 (통합)**: Self-Attention의 Q·K·V 연산을 시각화하고, 그 결과를 토대로 LLM이 어떻게 다음 토큰을 생성하는지 코드로 직접 구현한다.

---

## 부록 | 관찰 과제·혼자 점검 모범 답안

이 부록은 교육생이 실습과 이론을 끝낸 뒤 스스로 이해를 확인할 수 있도록 만든 해설이다.  
정답을 그대로 외우기보다, **왜 그런 답이 나오는지 설명할 수 있는지**를 기준으로 활용하는 것이 좋다.

### 관찰 과제 모범 답안

**1. `vocab_size`가 50,257인 이유는?**

GPT-2 토크나이저는 바이트 단위 BPE를 사용한다. 그래서 기본적으로 256개의 바이트 표현을 포함하고, 여기에 학습 과정에서 얻은 BPE 병합 결과가 더해져 전체 어휘가 구성된다. GPT-2의 경우 최종 어휘 수가 50,257개이며, 실무적으로는 "약 5만 개 규모의 서브워드 어휘"라고 이해하면 충분하다. 중요한 점은 이 구조 덕분에 처음 보는 단어도 바이트와 서브워드 조합으로 표현할 수 있다는 것이다.

**2. 한국어 텍스트를 입력했을 때 토큰이 어떻게 분리되는지 확인**

한국어는 GPT-2가 영어 중심으로 학습된 토크나이저를 사용하기 때문에, 영어보다 더 잘게 쪼개지거나 바이트 수준에 가까운 형태로 분해되는 경우가 많다. 즉, 영어 단어처럼 깔끔하게 하나 또는 두 개의 서브워드로 나뉘기보다, 여러 조각으로 잘릴 수 있다. 이 관찰을 통해 교육생은 "토크나이저의 성능과 효율은 학습 데이터와 언어 특성에 영향을 받는다"는 점을 이해할 수 있다.

**3. `hidden_states[0]`과 `hidden_states[-1]`의 shape 비교**

두 텐서의 shape는 보통 같다. 예를 들어 `(1, seq_len, 768)`처럼 배치 크기, 토큰 길이, 임베딩 차원은 동일하게 유지된다. 하지만 의미는 다르다.

- `hidden_states[0]`: 임베딩과 위치 정보가 합쳐진 초기 표현
- `hidden_states[-1]`: 여러 Transformer 블록을 거치며 문맥 정보가 반영된 최종 표현

즉, **shape는 같아도 내부 값의 의미와 정보량은 달라진다**.

### 혼자 점검하기 모범 답안

**1. Transformer가 RNN보다 병렬 처리에 유리한 이유는 무엇인가?**

RNN은 앞 시점의 계산 결과가 다음 시점 계산에 직접 연결되므로 토큰을 순서대로 처리해야 한다. 반면 Transformer는 Self-Attention을 사용해 문장 전체 토큰을 한 번에 행렬 연산으로 처리할 수 있다. 그래서 GPU 같은 병렬 연산 장치에 훨씬 잘 맞는다.

**2. Positional Encoding이 없으면 모델에 어떤 문제가 생기는가?**

Transformer는 기본적으로 토큰 집합을 동시에 보기 때문에, 위치 정보를 별도로 주지 않으면 단어 순서를 구분하기 어렵다. 예를 들어 `dog bites man`과 `man bites dog`는 같은 단어를 갖지만 순서가 바뀌면 의미가 달라진다. Positional Encoding은 이 순서 정보를 벡터에 추가해 모델이 문장 구조를 이해하게 만든다.

**3. Self-Attention에서 Q, K, V는 각각 어떤 역할을 하는가?**

- Q(Query): 현재 토큰이 어떤 정보를 찾고 있는지 나타내는 벡터
- K(Key): 각 토큰이 어떤 특징을 가지고 있는지 나타내는 벡터
- V(Value): 실제로 전달할 내용이 담긴 벡터

현재 토큰의 Q와 다른 토큰들의 K를 비교해 중요도를 계산하고, 그 가중치로 V를 합쳐 새로운 표현을 만든다. 즉, Q는 "질문", K는 "색인", V는 "실제 내용"으로 이해하면 된다.

**4. Multi-Head Attention은 Head가 1개인 경우보다 무엇이 좋은가?**

Head가 1개뿐이면 토큰 관계를 한 가지 시선으로만 해석하게 된다. 하지만 실제 문장에는 문법 관계, 의미 관계, 장거리 연결처럼 여러 패턴이 동시에 존재한다. Multi-Head Attention은 여러 Head가 서로 다른 관계를 병렬로 포착하게 해, 더 풍부한 문맥 표현을 만들 수 있다.

**5. GPT가 미래 토큰을 보지 못하도록 막는 이유는 무엇인가?**

GPT는 이전 문맥만 보고 다음 토큰을 예측하는 모델이기 때문이다. 만약 미래 토큰을 미리 볼 수 있다면, 생성해야 할 정답을 이미 참고하는 셈이 되어 학습 목표가 무너진다. 그래서 Causal Mask를 사용해 현재 위치보다 오른쪽의 토큰을 참조하지 못하게 만든다. 이 덕분에 학습 방식과 실제 생성 방식이 일치한다.

### 활용 팁

이 부록은 강사가 정답을 읽어주는 용도보다, 교육생이 아래 순서로 복습할 때 가장 효과적이다.

1. 먼저 스스로 답을 말하거나 적어 본다.
2. 그다음 모범 답안과 비교한다.
3. 표현이 달라도 핵심 개념이 맞으면 이해한 것으로 본다.
4. 설명이 막히는 질문은 다시 이론 또는 실습 코드로 돌아가 확인한다.